# Turbulent box tutorial

## Motivation

Turbulence is a very important component of the ISM. It is however not always easy to self-consistently model all sources driving turbulence. It is therefore often convenient to model it through the so-called Orstein-Uhlenbeck process, which apply a evolving acceleration computed in the Fourier space.
RAMSES includes a module that provides such a driving. Before starting the tutorial, it's a food idea to review the [RAMSES documentation](https://ramses-organisation.readthedocs.io/en/latest/wiki/TurbulenceDriving.html) discussing this module.

## Instalation

The tools we use are RAMSES and Osyris. More information on the codes here:
- https://github.com/nvaytet/osyris
- https://github.com/ramses-organisation/ramses

If you have followed the [general setup tutorial](https://ramses-tutorials.readthedocs.io/en/latest/Setup/general_requirements.html), osyris should already be installed. Otherwise, simply do

In [ ]:
%pip install osyris

You can also install it by cloning directly the source. In this case, please follow the instruction on the GitHub webpage (https://osyris.readthedocs.io/en/latest/installation.html).

To download RAMSES, do in your terminal

In [ ]:
%%bash
git clone https://github.com/ramses-organisation/ramses.git

To compile ramses, you need at least a fortran compiler, either fortran (GNU compiler) or ifort (Intel compiler). You can get more information on the GNU compiler here:
https://gcc.gnu.org/fortran/
https://fortran-lang.org/learn/os_setup/install_gfortran

>  **To run the turbulence driving simulations, you also need the FFTW library. Please follow what is explained in the file ramses/turb/README.**

## Run RAMSES

Once you downloaded RAMSES, go in the main directory (`cd ramses`)

### You can first run the test suite:

For each test, there is first a compilation (using the config.txt file of each test), then the code runs (using the namelist files .nml) and finally results are plotted using the `plot-test-name.py` routines.

There are various options for the test suite:    
```bash
#      - Run the suite in parallel (on 4 cpus):
              ./run_test_suite.sh -p 4
#      - Do not delete results data:
             ./run_test_suite.sh -d
#      - Run in verbose mode:
              ./run_test_suite.sh -v
#      - Select test number (for tests 3 to 5, and 10):
              ./run_test_suite.sh -t 3-5,10
#      - Run all tests in mhd directory:
              ./run_test_suite.sh -t mhd
#      - Run quick test suite:
              ./run_test_suite.sh -q
```
The option `-d` is quite useful if you want to work more on the data than what is proposed in the test suite. I would also recommend to use the `-y` option to use python for visualisation.

The turbulence driving test is the test number [16]. You will find in the directory `tests/turb/driving`  the routine plot-driving.py that can be an alternative to the use of osyris. In particular, you can produce column density maps that could be applied to the tools you learned last week!

Of course, you can also play with all the other tests! Note that you can find information on the namelist parameters you can play with here: https://ramses-organisation.readthedocs.io/en/latest/wiki/Runtime_Parameters.html

In [ ]:
%%bash
cd ramses/tests
./run_test_suite.sh -t turb -p 4 

This will run the test suite for the turbulence test case with 4 MPI processes.

### You can then compile directly ramses

Go to the bin directory and update the Makefile: NDIM=3 & USE_TURB=1 

_(note that this corresponds to the FLAGS indicated in the file config.txt of the turbulence driving test)._

**Then**, still in the bin directory


In [ ]:
%%bash
cd ramses/bin
make clean
make NDIM=3 USE_TURB=1 MPI=1 

You should have a file ramses3d that has been created (if not, double check that you updated the Makefile correctly).

You can then run ramses. 
Start by copying the executable you just build and the namelist from the tests directory in the present directory.
If you have MPI installed you can run on 4 processors: `mpirun -np 4 ramses3d driving.nml`

In [ ]:
%%bash 
mkdir isothermal
cp ramses/bin/ramses3d isothermal
cp ramses/tests/turb/driving/driving.nml isothermal

In [ ]:
%%bash
cd isothermal
./ramses3d driving.nml 

You can now play with the namelist parameters. You can change the resolution (`levelmin=5`, `levelmax=5` for a 32^3 resolution simulation).
You can add more outputs using the parameter foutput in &OUTPUT_PARAMS


```config
&OUTPUT_PARAMS
noutput=2
tout=0.0,0.5
foutput=25 ! Output every 25 steps
/
```



Enjoy!

## Use Osyris to visualise the data

You can check the documentation of osyris [here](https://osyris.readthedocs.io/en/stable/index.html).

### Plot a density slice


Modify the code below to 
- Add an extra layer with the velocity as arrows
- Change the spatial units to pc (you can use the parameter `dx`)
- Change the density unit to solar masses/pc^2
- Change the colormap to inferno

The final output should look like that:

![expected_slice](slice.png)

In [ ]:
import osyris
import numpy as np
import matplotlib.pyplot as plt

data = osyris.RamsesDataset(2, path="isothermal").load() # load output number 2. 
center = osyris.Vector(x=2, y=2, z=2, unit="pc") # Choose the center of the box

osyris.map(
    data["mesh"].layer("density"),  # layer 1 : density
    norm="log", origin=center, direction="z")

data["mesh"]["temperature"] = (data["mesh"]["pressure"] /  data["mesh"]["density"] /  (1* osyris.units("k_B")) * (1.4 * osyris.units("m_p"))).to("K")

osyris.map(
    data["mesh"].layer("temperature"),  # layer 1 : temperature
    norm="log", origin=center, direction="z", cmap="Reds_r")

### Plot distributions

You can also directly access the data and plot basic distribution information. Here is an example for a density PDF. You can adapt this to plot the velocity PDF and a phase plot.

In [ ]:
rho = data["mesh"]["density"].to("g/cm^3").values
rhomean = np.mean(rho) # works because uniform grid
n =  (data["mesh"]["density"] / (1.4 * osyris.units("m_p"))).to('1/cm^3').values

plt.figure()
plt.hist(np.log10(n), bins=100, histtype="step", lw=2)
plt.yscale("log")
plt.xlabel("log(n)")
plt.ylabel("DF")

## Build a movie

Adapt the namelist to tell rames to output data to make a movie. Here is an example you can start form. Don't forget to adapt it.

```fortran

&MOVIE_PARAMS
movie=.true.
imov=0
tstartmov=0.
tendmov=50
imovout=50
nw_frame=1000
nh_frame=1000
levelmax_frame=7
xcentre_frame=100.0,0.,0.,0.,100.0,0.,0.,0.,100.0,0.,0.,0.
ycentre_frame=100.0,0.,0.,0.,100.0,0.,0.,0.,100.0,0.,0.,0.
zcentre_frame=100.0,0.,0.,0.,100.0,0.,0.,0.,100.0,0.,0.,0.
deltax_frame=200.0,0.,200.0,0.,200.0,0.
deltay_frame=200.0,0.,200.0,0.,200.0,0.
deltaz_frame=200.0,0.,10.0,0.,200.0,0.
proj_axis='zzx'
movie_vars_txt = 'dens','temp'
/
```

One way to analyse it is to use RAM (for Rames Movie Maker). You can download it with:

In [ ]:
%%bash
git clone https://github.com/nbrucy/ram

You will need to modify the file ram3 to point towards a valid location of ffmpeg, eg. `/usr/bin/ffmpeg`.
Then **rerun** the simulation. RAM parameters a set in a `config.ini` file. Read it, modify it if necessary and you can generate the movie with

In [ ]:
%%bash
python ram/ram3.py

## More realistic thermodynamics

Right now our simulation are isothermal. Ramses include a module to compute the cooling function of typical ISM from it's thermal and density properties, based on Audit & Hennebelle 2005. In a new folder, copy the ramses executable and the namelist.

You may want to try different initial densities to develop an actual biphasic ISM. In particular, it could be interesting to choose initial conditions such that the gas the gas in initially in the unstable phase.
You may also want to take a larger box size, by changing `boxlen` and the definition of the region in `INIT_PARAMS`.
Another parameter you may want to choose carefully is the driving strength `turb_rms`.


Finally re-run the isothermal simulations with these new initial conditions so you can compare the two.



## What about magnetic field?

The next step is to add the magnetic field. For this you will need to 
- recompile the code with SOLVER=mhd
- Modify the namelist: the riemann solver needs to be set to `hlld`, as well as the `riemann2d` parameter.
- Set up the initial conditions for the magnetic field, also in the namelist. Check the documentation to see how to do that. Beware of the units.


You can then use Osyris to visualise the magnetic field. Some interesting diagnostic are the B-rho relation, and you can compare the PDF with the previous simulation.

## Further analysis

It's now up to you to decide which further analysis you want to carry on. To make things a bit more intersting, it's worth running the simulations with a bigger resolution (eg. at level 7, that $128^3$)

Here is a list of interesting quantities you may want to study and compare:
- Velocity dispersion
- PDF of density / velocity
- 2D and 3D power spectrum of density / velocity
- Helmotz decomposition of the field
- Vorticity and magnetic field generation
- Non-gaussian segmentation
- High order structure functions
- Alignement of the magnetic field and the density structures
- something new?